<a href="https://colab.research.google.com/github/abdul-sgf3/ProyekAkhir_5A_Adult_Census_Income/blob/main/02_Preprocessing(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [2]:
from google.colab import files

uploaded = files.upload()  # Pilih file adult.data
FILE_PATH = "adult.data"

columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"
]

df = pd.read_csv(FILE_PATH, header=None, names=columns, skipinitialspace=True)
print("Ukuran data awal (baris, kolom):", df.shape)
display(df.head())

Saving adult.data to adult.data
Ukuran data awal (baris, kolom): (32561, 15)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
# 1. Membersihkan spasi dan mengubah tanda '?' menjadi NaN
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

df = df.replace("?", np.nan)

print("Total nilai kosong sebelum penanganan:", df.isnull().sum().sum())
print("\nRincian missing value per kolom:")
display(df.isna().sum().sort_values(ascending=False))

# 2. Menghapus baris duplikat
jumlah_duplikat = df.duplicated().sum()
print("\nJumlah baris duplikat:", jumlah_duplikat)

df = df.drop_duplicates().copy()
print("Ukuran setelah hapus duplikat:", df.shape)

Total nilai kosong sebelum penanganan: 4262

Rincian missing value per kolom:


,0
occupation,1843
workclass,1836
native_country,583
fnlwgt,0
education,0
education_num,0
age,0
marital_status,0
relationship,0
sex,0



Jumlah baris duplikat: 24
Ukuran setelah hapus duplikat: (32537, 15)


In [4]:
# Hanya memilih 7 kolom fitur yang tercantum di Bagian C Proposal (tanpa kolom target)
selected_features = [
    "age",
    "workclass",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "hours_per_week"
]

X = df[selected_features].copy()
print("Kolom fitur yang digunakan:", X.columns.tolist())
print("Ukuran dataset fitur:", X.shape)

Kolom fitur yang digunakan: ['age', 'workclass', 'education_num', 'marital_status', 'occupation', 'relationship', 'hours_per_week']
Ukuran dataset fitur: (32537, 7)


In [5]:
numeric_features = ["age", "education_num", "hours_per_week"]
categorical_features = ["workclass", "marital_status", "occupation", "relationship"]

print("Fitur numerik:", numeric_features)
print("Fitur kategorikal:", categorical_features)

Fitur numerik: ['age', 'education_num', 'hours_per_week']
Fitur kategorikal: ['workclass', 'marital_status', 'occupation', 'relationship']


In [6]:
# Pipeline untuk numerik: Imputasi median + StandardScaler
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline untuk kategorikal: Imputasi modus + One-Hot Encoding
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

# Menjalankan preprocessing
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

processed_df = pd.DataFrame(
    X_processed,
    columns=feature_names,
    index=X.index
)

print("Ukuran data setelah preprocessing:", processed_df.shape)
display(processed_df.head())

Ukuran data setelah preprocessing: (32537, 38)


,num__age,num__education_num,num__hours_per_week,cat__workclass_Federal-gov,cat__workclass_Local-gov,cat__workclass_Never-worked,cat__workclass_Private,cat__workclass_Self-emp-inc,cat__workclass_Self-emp-not-inc,cat__workclass_State-gov,...,cat__occupation_Protective-serv,cat__occupation_Sales,cat__occupation_Tech-support,cat__occupation_Transport-moving,cat__relationship_Husband,cat__relationship_Not-in-family,cat__relationship_Other-relative,cat__relationship_Own-child,cat__relationship_Unmarried,cat__relationship_Wife
0,0.030390,1.134777,-0.035664,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.836973,1.134777,-2.222483,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,-0.042936,-0.420679,-0.035664,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1.056950,-1.198407,-0.035664,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,-0.776193,1.134777,-0.035664,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [7]:
print("Total missing value setelah preprocessing:", processed_df.isna().sum().sum())
print("Jumlah duplikat setelah preprocessing:", processed_df.duplicated().sum())

OUTPUT_PATH = "adult_preprocessed_kmeans.csv"
processed_df.to_csv(OUTPUT_PATH, index=False)
print(f"File berhasil disimpan: {OUTPUT_PATH}")

# Download hasil ke komputer
files.download(OUTPUT_PATH)

Total missing value setelah preprocessing: 0
Jumlah duplikat setelah preprocessing: 7467
File berhasil disimpan: adult_preprocessed_kmeans.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>